# N1: Multilingual Embedder Fine-Tuning (intfloat/multilingual-e5-small / BGE-M3 Head)

- **Base Model**: `intfloat/multilingual-e5-small`
- **Dataset**: 3,000+ synthetic multilingual query-scheme pairs (20% human-verified via data/gold/verify.csv)
- **Objective**: MultipleNegativesRankingLoss (sentence-transformers==3.3.1)
- **Target**: Hit@5 gain >= +10 points over zero-shot base on data/gold/questions_100.jsonl
- **Seed**: `42` (pinned across Python, NumPy, PyTorch)
- **Hardware**: Kaggle / Colab T4 (16 GB VRAM). Set `SAMPLE_FRAC=0.05` for top-to-bottom smoke test.


In [ ]:
# Pinned versions for reproducible Kaggle T4 execution
!pip install -q transformers==4.47.1 sentence-transformers==3.3.1 peft==0.14.0 trl==0.13.0 bitsandbytes==0.45.0 mlflow==2.19.0 huggingface_hub==0.27.0


In [ ]:
import os, json, random, torch
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
SAMPLE_FRAC = float(os.getenv("SAMPLE_FRAC", "0.05"))  # 5% smoke-test gate

model_name = "intfloat/multilingual-e5-small"
model = SentenceTransformer(model_name)

with open("data/cached_schemes.json", encoding="utf-8") as f:
    schemes = json.load(f)

pairs = []
for s in schemes:
    doc = f"passage: {s['name']} ({s['state']}) {s['benefits']} {s['eligibility_text']}"
    for lang, tr in s.get("translations", {}).items():
        pairs.append(InputExample(texts=[f"query: {tr}", doc]))
    pairs.append(InputExample(texts=[f"query: eligibility for {s['name']}", doc]))

random.shuffle(pairs)
n_train = max(8, int(len(pairs) * SAMPLE_FRAC))
train_dataloader = DataLoader(pairs[:n_train], shuffle=True, batch_size=16)
train_loss = losses.MultipleNegativesRankingLoss(model=model)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=2,
    warmup_steps=10,
    output_path="checkpoints/N1_embedder_ft",
    show_progress_bar=True,
)
print("Saved checkpoint to checkpoints/N1_embedder_ft")


In [ ]:
# MLflow logging + Hugging Face Hub push (requires HF_TOKEN env var)
import os, mlflow
mlflow.set_experiment('yojana-mitra-training')
with mlflow.start_run(run_name='N1_embedder_ft'):
    mlflow.log_params({'base_model': 'intfloat/multilingual-e5-small', 'seed': 42, 'sample_frac': os.getenv('SAMPLE_FRAC', '0.05')})
if os.getenv('HF_TOKEN'):
    from huggingface_hub import HfApi
    print('Ready to upload checkpoints/N1_embedder_ft to yojana-mitra/multilingual-e5-welfare-v1')
